In [1]:
!pip install datasets --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
from datasets import load_dataset

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

In [8]:
import json
import os
from datasets import load_dataset

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
OUTPUT = os.path.join(BASE_DIR, "data", "raw", "dailydialog_converted.jsonl")

# Primary: parquet-converted revision, bypasses the broken zip-download script
try:
    dataset = load_dataset(
        "li2017dailydialog/daily_dialog",
        split="train",
        revision="refs/convert/parquet",
        trust_remote_code=True
    )
except Exception as e:
    print(f"Primary load failed ({e}), falling back to community mirror...")
    dataset = load_dataset("Akhil391/daily_dialog", split="train")

print(f"Loaded {len(dataset)} dialogues")
print("Sample raw entry:", dataset[0])

MAX_CHARS = 2000  # rough proxy for ~512 tokens combined; tune after checking token counts

kept, dropped = 0, 0

with open(OUTPUT, "w", encoding="utf-8") as out:
    for dialogue in dataset:
        turns = [t.strip() for t in dialogue["dialog"] if t.strip()]
        if len(turns) < 2:
            continue

        for i in range(1, len(turns)):
            instruction = turns[i - 1]
            response = turns[i]

            # last 1 exchange (2 turns) before the instruction turn, if available
            context_block = ""
            if i >= 3:
                ctx_user = turns[i - 3]
                ctx_asst = turns[i - 2]
                context_block = f"### Context:\nUser: {ctx_user}\nAssistant: {ctx_asst}\n\n"

            text = (
                f"{context_block}"
                f"### Instruction:\n{instruction}\n\n"
                f"### Response:\n{response}"
            )

            if len(text) > MAX_CHARS:
                dropped += 1
                continue

            out.write(json.dumps({"text": text}) + "\n")
            kept += 1

print(f"Kept: {kept}, Dropped (oversized): {dropped}")

default/train/0000.parquet:   0%|          | 0.00/3.61M [00:00<?, ?B/s]

C:\Users\MIND & MATTER\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MIND & MATTER\.cache\huggingface\hub\datasets--li2017dailydialog--daily_dialog. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


0000.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/331k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loaded 11118 dialogues
Sample raw entry: {'dialog': ['Say , Jim , how about going for a few beers after dinner ? ', ' You know that is tempting but is really not good for our fitness . ', ' What do you mean ? It will help us to relax . ', " Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? ", " I guess you are right.But what shall we do ? I don't feel like sitting at home . ", ' I suggest a walk over to the gym where we can play singsong and meet some of our friends . ', " That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . ", ' Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too . ', " Good.Let ' s go now . ", ' All right . '], 'act': [3, 4, 2, 2, 2, 3, 4, 1, 3, 4], 'emotion': [0, 0, 0, 0, 0, 0, 4, 4, 4, 4]}
Kept: 76051, Dropped (oversized): 1
